# Ephemeris Optimization Analysis With Polynomial Basis

This notebook performs the following operations, to reproduce the results in the paper [Cortinovis, M., Iiyama, K., and Gao, G.,"Satellite Ephemeris Parameterization Methods to Support Lunar Positioning, Navigation, and Timing Services"](https://navi.ion.org/content/71/4/navi.664) 

- Propagate orbit in the lupnt simulator
- Set up optimization problem and data collection options
- Solve optimization problem with chosen basis, orders, and time intervals
- Evaluate ephemeris error and dunmp data to JSON file
- Post processing

## Part1: Orbit Propagation
In this first part, we will propagate the orbit using the LuPNT propagator

### 1.1 Choose Orbit and Setup Options
Elliptical Lunar Frozen Orbit and Low Lunar Orbit available

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib
import plotly.graph_objects as go
import matplotlib.pyplot as plt

import pylupnt as pnt

from ex_ephemeris.qllo import qllo_opt
import pylupnt.ephemeris as eph
from ex_ephemeris.ex_ephemeris_utils import *

Initial orbit elements setup

In [ ]:
orbit = "LLO"

# time
t0 = pnt.gregorian2time(2022, 1, 2, 0, 0, 0)

if orbit == "LLO":
    print("LLO")
    t_end = 6.0 * 3600 + 0.01

    a_init = 1850.0
    e_init = 0.05

    coe0 = qllo_opt(a0=a_init, e0=e_init)
    rv0_op = pnt.classical_to_cart(coe0, pnt.GM_MOON)
    rv0_mi = pnt.convert_frame(t0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)

elif orbit == "ELFO":
    print("ELFO")
    t_end = 30.0 * 3600 + 0.01
    # Classical Orbital Elements (COE)
    a = 6541.4e3  # [m] Semi-major axis
    e = 0.6000  # [--] Eccentricity
    i = 56.2 * pnt.RAD  # [deg] Inclination
    O = 0.00 * pnt.RAD  # [deg] Right ascension of the ascending node
    w = 90.0 * pnt.RAD  # [deg] Argument of perigee
    M = 0.00 * pnt.RAD  # [deg] Mean anomaly

    coe0 = np.array([a, e, i, O, w, M])  # in op frame
    rv0_op = pnt.classical_to_cart(coe0, pnt.GM_MOON)
    rv0_mi = pnt.convert_frame(t0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)

coe_mi = pnt.cart_to_classical(rv0_mi, pnt.GM_MOON)
T_orbit = 2 * np.pi * np.sqrt(coe_mi[0] ** 3 / pnt.GM_MOON)

dt_step = 1.0
dt_prop = 1.0

# print the orbital elements
print("Initial state (OP Frame):")
print(" a [m]:", coe0[0])
print(" e [-]:", coe0[1])
print(" i [deg]:", coe0[2] * pnt.DEG)
print(" O [deg]:", coe0[3] * pnt.DEG)
print(" w [deg]:", coe0[4] * pnt.DEG)
print(" M [deg]:", coe0[5] * pnt.DEG)
print(" ")
print("Initial state (CI Frame):")
print("a [m]:", coe_mi[0])
print("e [-]:", coe_mi[1])
print("i [deg]:", coe_mi[2] * pnt.DEG)
print("O [deg]:", coe_mi[3] * pnt.DEG)
print("w [deg]:", coe_mi[4] * pnt.DEG)
print("M [deg]:", coe_mi[5] * pnt.DEG)

### 1.2 Propagate Orbit using Multi-body Dynamics
The list of perturbations are as follows
- 50x50 Moon gravity
- Third body: Earth, Sun

For integration, we use the RKF-45 integrator (adaptive stepsize) with RelTol=1e-12, AbsTol=1e-12

In [ ]:
dyn = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-12, reltol=1e-12))

dyn.add_body(pnt.Body.Moon(50, 50))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_time_step(dt_prop)
dyn.set_frame(pnt.MOON_CI)

The actual propagation takes place here

In [ ]:
tspan = t0 + np.arange(0, t_end * 1.5, dt_step)
rv_prop_mi = dyn.propagate(rv0_mi, t0, tspan, progress=True)  # in Moon Inertial frame
rv_prop_op = pnt.convert_frame(
    t0 * np.ones(np.size(tspan)), rv_prop_mi, pnt.MOON_CI, pnt.MOON_OP, rotate_only=True
)  # in OP frame
rv_prop_pa = pnt.convert_frame(
    tspan, rv_prop_mi, pnt.MOON_CI, pnt.MOON_PA
)  # in Planetary frame
rv_prop_me = pnt.convert_frame(
    tspan, rv_prop_mi, pnt.MOON_CI, pnt.MOON_ME
)  # in Planetary frame

### 1.3 3-D Orbit Visualization

In [ ]:
# visualize orbit
orb_plot = np.zeros((4, len(tspan), 6))
orb_plot[0, :, :] = rv_prop_mi
orb_plot[1, :, :] = rv_prop_op
orb_plot[2, :, :] = rv_prop_pa
orb_plot[3, :, :] = rv_prop_me

fig = go.Figure()
pnt.plot.plot_body(fig, pnt.MOON)
pnt.plot.plot_orbits(
    fig,
    orb_plot,
    color=["blue", "red", "green", "yellow"],
)
pnt.plot.set_view(fig, azimuth=-130, elevation=10, zoom=2.5)
fig.update_layout(width=400, height=400)
# set labels (mci, PA)
text_fig = """
<b>Satellite orbit</b><br>
Red: Moon OP<br>
Blue: Moon CI<br>
Green: Moon PA<br>
Yellow: Moon ME
"""

fig.add_annotation(
    text=text_fig, **dict(x=0.01, y=0.98, align="left", xanchor="left", showarrow=False)
)
fig.show()

### 1.4 Osculating Orbit Elements Visualization
Next, we plot the osculating orbit elements. We see there is a secular drift in eccentricity, inclination, and Omega


In [ ]:
# Plot
coe_op = pnt.cart_to_classical(rv_prop_op, pnt.GM_MOON)
labels = [
    "Semi-major axis [m]",
    "Eccentricity [-]",
    "Inclination [deg]",
    "Right Asc. [deg]",
    "Arg. of Periapsis [deg]",
    "Mean Anomaly [deg]",
]

fig = plt.figure(figsize=(8, 6))
matplotlib.rcParams.update({"font.size": 12})
x = (tspan - t0) / pnt.SECS_HOUR
if orbit == "LLO":
    plt.suptitle("Orbital Elements (Quasi-Frozen LLO)")
elif orbit == "ELFO":
    plt.suptitle("Orbital Elements (ELFO)")

for i in range(6):
    plt.subplot(3, 2, i + 1)
    y = coe_op[:, i] if i < 2 else coe_op[:, i] * pnt.DEG
    plt.plot(x, y)
    plt.xlabel("Hours past " + pnt.time2gregorian_string(t0) + " TAI")
    plt.ylabel(labels[i])
    plt.grid()
    if i < 5:
        plt.xlim(x[0], x[-1])
    else:
        plt.xlim(x[0], x[100])
plt.tight_layout()
plt.show()

### 1.5 Lunar Orientation Angle Visualization
Finally, we plot the three lunar orientation angles ($\phi$, $\theta$, $\psi$) and its rates. In short term, their changes are nearly linear.

In the figure below, the lunar frames are shown in ME frame, but in this script we are computing the angles to the PA frame.



In [ ]:
# Extract the lunar orientation angles
angles = np.zeros((len(tspan), 6))
for i, t in enumerate(tspan):
    angles[i, :] = pnt.get_lunar_orientation_angles(t)  # MCI to PA angles

# Plot
# phi, theta, psi, phi_dot, theta_dot, psi_dot
labels = [
    "Phi [deg]",
    "Theta [deg]",
    "Psi [deg]",
    "Phi_dot [deg/s]",
    "Theta_dot [deg/s]",
    "Psi_dot [deg/s]",
]

fig = plt.figure(figsize=(12, 6))
matplotlib.rcParams.update({"font.size": 12})
x = (tspan - t0) / pnt.SECS_DAY
plt.suptitle("Lunar Orientation Angles")
for i in range(6):
    plt.subplot(3, 2, i + 1)
    y = angles[:, i] * pnt.DEG
    if i <= 2:
        y = np.mod(y, 360)
    plt.plot(x, y)
    plt.xlabel("Days past " + pnt.time2gregorian_string(t0) + " TAI")
    plt.ylabel(labels[i])
    plt.grid()
    plt.xlim(x[0], x[-1])

plt.tight_layout()
plt.show()

## 2. Ephemeris Optimization Setup
In this section, we will define parameters and organize the orbit and angles data as dataframes to be processed for LuPNT ephemeris generator

### 2.1 Data Parsing
Generate a pandas dataframe from your propagated orbit

In [ ]:
# convert to dataframes
import pandas as pd

time = tspan - t0


def construct_data(t_end):
    # Inertial states
    r_MCI = pd.DataFrame(rv_prop_mi[:, :3], columns=["x", "y", "z"])
    v_MCI = pd.DataFrame(rv_prop_mi[:, 3:], columns=["v_x", "v_y", "v_z"])
    df_MCI = pd.concat([r_MCI, v_MCI], ignore_index=True, axis=1)
    df_MCI.columns = ["x", "y", "z", "v_x", "v_y", "v_z"]
    r_MCI.insert(0, "time", time)
    v_MCI.insert(0, "time", time)
    df_MCI.insert(0, "time", time)

    # MCMF states
    r_MCMF = pd.DataFrame(rv_prop_pa[:, :3], columns=["x", "y", "z"])
    v_MCMF = pd.DataFrame(rv_prop_pa[:, 3:], columns=["v_x", "v_y", "v_z"])
    df_MCMF = pd.concat([r_MCMF, v_MCMF], ignore_index=True, axis=1)
    df_MCMF.columns = ["x", "y", "z", "v_x", "v_y", "v_z"]
    r_MCMF.insert(0, "time", time)
    v_MCMF.insert(0, "time", time)
    df_MCMF.insert(0, "time", time)

    # orbital elements
    coe_op_f = coe_op
    # convert mean anomaly to true anomaly
    true_anom = pnt.mean_to_true_anomaly(coe_op_f[:, 5], coe_op_f[:, 1])
    # convert from [-pi, pi] to [0, 2pi]
    true_anom = np.mod(true_anom, 2 * np.pi)
    coe_op_f[:, 5] = true_anom

    df_OE = pd.DataFrame(coe_op_f, columns=["a", "e", "i", "O", "w", "f"])
    df_OE.insert(0, "time", time)

    # extract euler angles
    # phi, theta, psi, phi_dot, theta_dot, psi_dot

    eul_dict = {
        "time": time,
        "psi": np.mod(angles[:, 2], 2 * np.pi),
        "theta": np.mod(angles[:, 1], 2 * np.pi),
        "phi": np.mod(angles[:, 0], 2 * np.pi),
        "psi_dot": np.mod(angles[:, 5], 2 * np.pi),
        "theta_dot": np.mod(angles[:, 4], 2 * np.pi),
        "phi_dot": np.mod(angles[:, 3], 2 * np.pi),
    }
    df_angles = pd.DataFrame(eul_dict)

    # cut the data frames
    df_angles = df_angles[df_angles["time"] <= t_end + 0.001]

    return df_MCI, df_MCMF, df_angles, df_OE


df_MCI, df_MCMF, df_angles, df_OE = construct_data(t_end)

# construct dictionary
df_dict = {"MCI": df_MCI, "MCMF": df_MCMF, "angles": df_angles, "OE": df_OE}

### 2.2 Plot the output of each basis

In [ ]:
import pylupnt.ephemeris as eph
import matplotlib.pyplot as plt
import matplotlib

# Plot the polynomials for the region [-1, 1]
n_array_test = [1, 3, 5, 7, 9, 11]
fit_basis = ["polynomial", "cheby", "legendre", "fourier"]

n_basis = len(fit_basis)
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
num_points = 100

poly = eph.Polynomial()
cheb = eph.Chebyshev()
leg = eph.Legendre()
four = eph.Fourier()

fit_basis_inst = [poly, cheb, leg, four]
fit_basis_str = ["polynomial", "chebyshev", "legendre", "fourier"]

cmap = matplotlib.cm.get_cmap("viridis")

for i, basis in enumerate(fit_basis_inst):
    xs = np.linspace(-1, 1, num_points)
    ax = axs[i // 2, i % 2]
    for n in n_array_test:
        ys = np.zeros((num_points))
        for j, x in enumerate(xs):
            ys[j] = basis.generate_pol(n, x)

        # color scale between 0.2 and 0.8
        cscale = 0.2 + (0.8 - 0.2) * (n - n_array_test[0]) / (
            n_array_test[-1] - n_array_test[0]
        )
        ax.plot(xs, ys, label=f"n={n}", color=cmap(cscale))

    ax.set_title(fit_basis_str[i])
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.grid()
    # axis equal
    ax.set_aspect("equal", adjustable="box")
    ax.legend()

plt.tight_layout()
plt.show()

### 2.3 Use Parameter Selection <-- Edit Here

In [ ]:
# choose how to fit
fit_basis = ["polynomial", "cheby", "legendre", "fourier"]

###-------------------------------BEGIN EDTIS-----------------------------------------------###
solve_basis = fit_basis[:3]  # choose basis to solve (fourier does not work)
angle_model = "linear"
anom_num = 31  # How many true anomalies to sample
n_array = [6, 8, 10, 12, 14]  # number of degrees
tinv_eval = 1  # Time interval for evaluating SISE constraints

data_dump = False
over_det = False
test = False  # test-mode: just run single case
rerun = True  # rerun the optimization

pos_req = 13.43e-3 / 6  # position requirement
vel_req = (1.2e-6) / 6  # velocity requirement
add_sise_constraint = True
fit_velocity = True
cheby_interp = True
###-------------------------------END EDITS-------------------------------------------------###


if test:
    if orbit == "LLO":
        t_intervals = [15.0 / 60 * 3600]
    elif orbit == "ELFO":
        t_intervals = [1.0 * 3600]
    n_array = [14]  # number of points
    anom_num = 3

else:
    if orbit == "LLO":
        t_intervals = np.array([5, 10, 15, 20, 30, 45, 60, 90, 120]) * 60
        # t_intervals = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.6, 0.8, 1.0]) * T_orbit
        # t_intervals = [10/60*3600, 20/60*3600, 36/60*3600, 72/60*3600, 108/60*3600]

    if orbit == "ELFO":

        # t_intervals = [1.0*3600, 2.0*3600, 4.0*3600, 8.0*3600, 12.0*3600]
        # t_intervals = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.6, 0.8, 1.0]) * T_orbit
        t_intervals = np.array([10, 30, 60, 90, 120, 180, 240, 360, 480]) * 60

        # round to nearest timestep
        t_intervals = np.round(t_intervals / dt_step) * dt_step
        print("Time intervals for SISE constraints:", t_intervals)

        #  t_intervals = [0.55*3600, 1.66*3600, 3.33*3600, 6.66*3600, 13.3*3600]
    # n_array = [6,8,10,12,14]   # number of points

# print configurations
print("Solve basis:", solve_basis)
print("Angle model:", angle_model)
print("Fit Order:", n_array)
print("Anomaly number:", anom_num)
print("Time interval for SISE constraints:", tinv_eval)

## Part3: Solve Optimization Problem
Finally, we solve the optimization problem. Note that this cell can take a long time (> 1 hr for ELFO) depending on the configuration of the cell before

In [ ]:
# supress warning
import warnings

warnings.filterwarnings("ignore")  # ignore warnings for ECOS

pos_idx = ["x", "y", "z"]
vel_idx = ["v_x", "v_y", "v_z"]
# list of dictionaries, ordered by the time interval of approximation
sise_dict_basis = dict()
sise_dict = [
    {key: [] for key in ["f0", "t0", "n", "sise_pos", "sise_vel", "coefficients"]}
    for t_idx in range(len(t_intervals))
]

if rerun:
    for basis in solve_basis:
        print(" ")
        print(f"Solving with {basis} basis")

        sise_dict = basis_fit_eval_ephemeris(
            df_dict,
            t_intervals,
            tinv_eval,
            anom_num,
            T_orbit,
            n_array,
            add_sise_constraint,
            basis,
            angle_model,
            pos_req,
            vel_req,
            fit_velocity,
            cheby_interp,
            plot_fig=test,
            use_parallel=True,
        )
        sise_dict_basis[basis] = sise_dict

print("Simulation Done!")

## Part 4: Save Data into JSON File

In [ ]:
import json
import os
import pickle
from os.path import join

filepath = join(pnt.LUPNT_OUTPUT_PATH, "ephemeris")
print(filepath)
os.makedirs(filepath, exist_ok=True)

print(sise_dict_basis)

if rerun:
    if orbit == "LLO":
        for basis in solve_basis:
            sise_dict = sise_dict_basis[basis]

            if basis == "polynomial":
                with open(join(filepath, "LLO_polynomial_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

            if basis == "cheby":
                with open(join(filepath, "LLO_cheby_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

            if basis == "legendre":
                with open(join(filepath, "LLO_legendre_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

        # store t_intervals into a pickle file
        with open(join(filepath, "LLO_t_intervals.pkl"), "wb") as outfile:
            pickle.dump(t_intervals, outfile)

        # store T_orbit into a pickle file
        with open(join(filepath, "LLO_T_orbit.pkl"), "wb") as outfile:
            pickle.dump(T_orbit, outfile)

    if orbit == "ELFO":
        for basis in solve_basis:
            sise_dict = sise_dict_basis[basis]

            if basis == "polynomial":
                with open(join(filepath, "ELFO_polynomial_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

            if basis == "cheby":
                with open(join(filepath, "ELFO_cheby_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

            if basis == "legendre":
                with open(join(filepath, "ELFO_legendre_extra.json"), "w") as outfile:
                    json.dump(sise_dict, outfile)

        # store t_intervals into a pickle file
        with open(join(filepath, "ELFO_t_intervals.pkl"), "wb") as outfile:
            pickle.dump(t_intervals, outfile)

        # store T_orbit into a pickle file
        with open(join(filepath, "ELFO_T_orbit.pkl"), "wb") as outfile:
            pickle.dump(T_orbit, outfile)

## 5. Post Processing

### 5.0 Load the Generated Data

In [ ]:
import matplotlib.pyplot as plt

orbit = "ELFO"  # <-------- Change here!   (ELFO, LLO)
basis_list = ["polynomial", "cheby", "legendre"]
sise_dicts = dict()

# Load and store data
for basis in basis_list:
    # load data for each coefficient
    filepath = join(pnt.LUPNT_OUTPUT_PATH, "ephemeris")
    json_name = f"{orbit}_{basis}_extra.json"

    # load the json file
    with open(join(filepath, json_name), "rb") as infile:
        sise_dicts[basis] = json.load(infile)

    if orbit == "LLO":
        # load t_intervals from pickle
        with open(join(filepath, "LLO_t_intervals.pkl"), "rb") as infile:
            t_intervals = pickle.load(infile)

        # load T_orbit from pickle
        with open(join(filepath, "LLO_T_orbit.pkl"), "rb") as infile:
            T_orbit = pickle.load(infile)

    if orbit == "ELFO":
        # load t_intervals from pickle
        with open(join(filepath, "ELFO_t_intervals.pkl"), "rb") as infile:
            t_intervals = pickle.load(infile)

        # load T_orbit from pickle
        with open(join(filepath, "ELFO_T_orbit.pkl"), "rb") as infile:
            T_orbit = pickle.load(infile)


# create output dir
os.makedirs("output/ex_ephemeris", exist_ok=True)

### 5.1 Error vs Fit Interval

In [ ]:
# Plot 1: x-axis: fit interval, y-axis: error -----------------------
for basis in basis_list:
    sise_dict = sise_dicts[basis]
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))

    for i, t in enumerate(t_intervals):
        n = sise_dict[i]["n"]
        f0 = sise_dict[i]["f0"]

        sise_pos = sise_dict[i]["sise_pos"]
        sise_vel = sise_dict[i]["sise_vel"]

        ax[0].scatter(t * np.ones(len(sise_pos)) / T_orbit, sise_pos)
        ax[1].scatter(t * np.ones(len(sise_pos)) / T_orbit, sise_vel)

    for i in range(2):
        ax[i].grid(True)
        ax[i].set_xlabel("Fit Interval (Orbits)")
        ax[i].set_xlim(0, 1)
        if i == 0:
            ax[i].set_ylabel("Position (3 sigma)")
            ax[i].set_ylabel("Error [m]")
        else:
            ax[i].set_ylabel("Velocity (3 sigma)")
            ax[i].set_ylabel("Error [mm/s]")

    plt.suptitle(f"Error vs Fit Interval ({orbit}, {basis})")
    plt.tight_layout()
    plt.savefig(f"output/ex_ephemeris/ErrorFitInterval_{orbit}_{basis}.pdf")
    plt.show()

### 5.2 Plot Valid Ephemeris Length

In [ ]:
# Plot 2: Valid Ephemeris Length -----------------------------------
pos_limit = 13.46
vel_limit = 1.20

for basis in basis_list:
    sise_dict = sise_dicts[basis]
    n_array = np.unique(sise_dict[0]["n"])
    fvals = np.unique(sise_dict[0]["f0"])
    print(fvals)
    n_f0 = len(fvals)
    n_n = len(n_array)
    n_tinv = len(t_intervals)

    max_tinv = np.zeros((n_f0, n_n))
    sise_pos_array = np.zeros((n_f0, n_tinv, n_n))
    sise_vel_array = np.zeros((n_f0, n_tinv, n_n))

    for i, t in enumerate(t_intervals):
        n = sise_dict[i]["n"]
        f0 = sise_dict[i]["f0"]
        sise_pos = sise_dict[i]["sise_pos"]
        sise_vel = sise_dict[i]["sise_vel"]

        for j, f in enumerate(fvals):
            fidxs = np.where(f0 == f)[0]
            satisfy_constraint = False

            for idx in fidxs:
                ni = np.where(n_array == n[idx])[0]
                sise_pos_array[j, i, ni] = sise_pos[idx]
                sise_vel_array[j, i, ni] = sise_vel[idx]

                if sise_pos[idx] <= pos_limit and sise_vel[idx] <= vel_limit:
                    max_tinv[j, ni] = t
                    satisfy_constraint = True
                    break

    # histogram plot
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, n in enumerate(n_array[::-1]):
        plt.bar(fvals, max_tinv[:, n_n - 1 - i] / T_orbit, width=5.0, label=f"n={n}")
    plt.grid(True, "minor")
    plt.grid(True, "major")
    plt.xlabel("True Anomaly")
    plt.ylabel("Max Fit Interval (Orbits)")
    plt.legend()
    plt.savefig("output/ex_ephemeris/max_fit_interval_{}_{}.pdf".format(orbit, basis))
    plt.title(f"Max Fit Interval vs True Anomaly ({orbit}, {basis})")

### 5.3 Plot Required Message Length
Finally, we solve the required message size 
We observe that the smaller number of bits are required for Chebyshev and Legendre polynomials, due to its recursive nature keeping the magnitude of the coefficients small

In [ ]:
# Compute the number of bits required for each coefficient
message_size = dict()

for basis in basis_list:
    sise_dict = sise_dicts[basis]
    message_size[basis] = dict()

    coeff_store = dict()

    # extract the coeffcients for each degree ad
    for i, t in enumerate(t_intervals):
        coeff_store[t] = dict()
        nvals = sise_dict[i]["n"]
        f0vals = sise_dict[i]["f0"]
        coeffs = sise_dicts[basis][i]["coefficients"]

        message_size[basis][t] = dict()

        for n in np.unique(nvals):
            coeff_store[t][n] = dict()
            message_size[basis][t][n] = 0
            for f in np.unique(f0vals):
                coeff_store[t][n][f] = dict()

        # create coeff store
        for j, f in enumerate(f0vals):
            coeff = coeffs[j]
            n = nvals[j]
            coeff_x = coeff[0]
            coeff_y = coeff[1]
            coeff_z = coeff[2]
            coeff_angle = coeff[3][:3]  # angle
            coeff_angle_rate = coeff[3][3:]

            # compute message size
            Nb = 0

            for l, param in enumerate(
                [coeff_x, coeff_y, coeff_z, coeff_angle, coeff_angle_rate]
            ):
                coeff_store[t][n][f][l] = param

        for k, n in enumerate(nvals):

            Nb = 0  # number of bits d f

            for l in range(5):
                if l < 3:
                    param_len = n + 1  # number of coefficients
                    params = np.zeros((param_len, len(np.unique(f0vals))))
                    for j, f in enumerate(np.unique(f0vals)):
                        params_tmp = coeff_store[t][n][f][l][
                            : n + 1
                        ]  # param_size x f0vals
                        # print("l: ", l, " n:", n, "f:", f, " params: ", params_tmp)
                        params[:, j] = params_tmp

                    # if max of the coefficients is zero (=no solution found), set bits to zero
                    if np.max(np.max(params)) == 0:
                        Nb = np.nan
                        break

                    pmax = np.max(
                        np.abs(params), axis=1
                    )  # take max over f0vals  (param_size x 1)
                    mag = np.array(
                        [10 ** (np.ceil(np.log10(pm))) for pm in pmax]
                    )  # digits
                    lsb = 1e-8  # least significant bit  1e-8 km/s = 0.01 mm/s
                elif l == 3:  # angle
                    mag = (2 * np.pi) * np.ones(3)
                    lsb = 1e-7  # least significant bit  rad
                elif l == 4:  # angle rate
                    mag = (1e-5) * np.ones(3)
                    lsb = 1e-15

                # compute the required number of bits
                bits_req = np.ceil(np.log10(mag / lsb) / np.log10(2)) + 1

                # print("l: ", l, " n:", n, " bits_req: ", bits_req)
                Nb += sum(bits_req)

            message_size[basis][t][n] = Nb

# plot the number of required bits
# x axis: degree of polynomial, y axis: number of bits
tinvs = t_intervals[:8]
print(t_intervals / 60)
fig, axs = plt.subplots(2, 4, figsize=(15, 8))

colors = ["b", "r", "g", "y", "k"]

for j, t in enumerate(tinvs):
    ax = axs[j // 4, j % 4]
    for i, basis in enumerate(basis_list):
        nbits = message_size[basis][t]
        ax.plot(list(nbits.keys()), list(nbits.values()), "o-", label=basis)
    ax.set_title("t={0:.1f} min".format(t / 60))
    ax.grid(True, "minor")
    ax.grid(True, "major")
    ax.set_xlabel("Degree of Polynomial")
    ax.set_ylabel("Number of Bits")
    ax.legend(loc="upper left")
    ax.set_xlim(5.8, 14.2)
    ax.set_ylim(500, 2500)

plt.tight_layout()
plt.savefig("output/ex_ephemeris/num_bits_{}.pdf".format(orbit))
plt.show()